# Lecture: Relationships Between Quantitative Features

Examine how two quantitative features vary together. Use scatterplots, correlation, and visual regression lines to describe relationships; then compare original and logarithmic scales to explore a curved pattern.


## Learning goals

- Create and interpret scatterplots using direction, form, strength, and unusual observations.
- Calculate and interpret Pearson correlation and simple-line R-squared in context.
- Use a regression plot as a visual summary of an approximately linear association.
- Use color and group-specific lines to explore additional categorical information, then compare model R-squared values.
- Recognize that a strong correlation does not guarantee a straight relationship or causation.
- Compare original and log-transformed measurements to see how a transformation changes the visible form of a relationship.


## Import the libraries


Import NumPy, Pandas, Matplotlib's `pyplot` component, and Seaborn using their conventional aliases.

**NumPy** provides numerical tools. We will use `np.where()` to create category labels and `np.log10()` to transform positive measurements.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Florida county income and life expectancy

The County Health Rankings & Roadmaps program assembles measures of population health and community conditions for U.S. counties. Economic resources can shape access to housing, food, education, health care, transportation, and safer environments. That motivates a genuine—but carefully limited—research question about income and health.


## The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can answer the question?
3. **Evidence:** What should we calculate and display?
4. **Check:** Does the evidence make sense in context and agree with the original records?
5. **Conclusion:** What claim is supported?
6. **Limitation:** What should we avoid concluding?


## Step 1 — Question

> **How was median household income related to life expectancy across Florida counties in the 2025 County Health Rankings release?**
>
> **Did this relationship depend on resident age or access to primary-care physicians?**

We expect that counties with greater income may also have greater life expectancy, but the relationship is not automatic. Counties with similar incomes can differ in demographics, environments, health-care access, public policy, and many other conditions.


## Discussion: predict the relationship and target

1. What direction do you predict for the relationship between income and life expectancy?

2. Which feature should be the target and which should be used as the predictor?


## Step 2 — Data

Read the fixed dataset. The FIPS code is an identifier rather than a quantity, so `dtype={"county_fips": "string"}` preserves leading zeros and prevents arithmetic from being applied to it.


In [ ]:
all_counties = pd.read_csv(
    "data/county_health_rankings_2025.csv",
    dtype={"county_fips": "string"}
)


Filter the national file to Florida before inspecting or modeling the data. `.loc` selects rows where `state` equals `"FL"`; `.copy()` creates an independent DataFrame. Each row is one county, so each county has equal weight regardless of population.


In [ ]:
county_health = all_counties.loc[all_counties["state"] == "FL"].copy()

county_health.shape


In [ ]:
county_health.head()


In [ ]:
county_health.tail()


In [ ]:
county_health.info()


In [ ]:
county_health.isnull().sum()


### Data dictionary

| Feature | Description |
| --- | --- |
| `county_fips` | Five-character federal county code |
| `county` | County or county-equivalent name |
| `state` | Two-letter state abbreviation |
| `median_household_income_usd` | Median annual household income in U.S. dollars |
| `life_expectancy_years` | Average expected years of life from birth |
| `age_65_plus_percent` | Percentage of residents age 65 or older |
| `primary_care_physicians_per_100000` | Primary-care physicians per 100,000 residents |

The **release year is not the measurement year**. County Health Rankings combines the most recently available source periods, which differ by measure. The documentation identifies the original sources and periods.


## Discussion: check the data

Complete the table using `.info()`, `.isnull().sum()`, and the data dictionary.

| Feature | Data type | Data storage type | Missing values |
| --- | --- | --- | ---: |
| `county_fips` |  |  |  |
| `county` |  |  |  |
| `state` |  |  |  |
| `median_household_income_usd` |  |  |  |
| `life_expectancy_years` |  |  |  |
| `age_65_plus_percent` |  |  |  |
| `primary_care_physicians_per_100000` |  |  |  |


### Prepare the comparison data

Use the same observations for the income scatterplot and the two colored group comparisons. Create `analysis` by dropping rows missing any of the four required measurements. Income and life expectancy are available for all 67 Florida counties, but two counties lack primary-care estimates, leaving 65 complete observations for consistent comparisons.

The full-column missing-value check above shows that only primary-care access is missing. Here, `.dropna()` therefore removes the same two counties without specifying a subset of columns. Keep the original `county_health` DataFrame. An unavailable measurement does not mean zero.


In [ ]:
analysis = county_health.dropna().copy()

analysis.shape


### Inspect the prepared values

Confirm that the selected features are complete and inspect their ranges before plotting. This initial inspection can flag missing or implausible values but cannot establish that every measurement is correct.


In [ ]:
analysis.isnull().sum()


In [ ]:
analysis.describe().round(1)


The minimum and maximum deserve more context than `.describe()` provides. Display the counties at the bottom and top of the income and life-expectancy distributions so we can ask whether the values and locations are geographically plausible.


In [ ]:
analysis.nsmallest(5, "median_household_income_usd")[
    ["county", "state", "median_household_income_usd"]
]


In [ ]:
analysis.nlargest(5, "median_household_income_usd")[
    ["county", "state", "median_household_income_usd"]
]


In [ ]:
analysis.nsmallest(5, "life_expectancy_years")[
    ["county", "state", "life_expectancy_years"]
]


In [ ]:
analysis.nlargest(5, "life_expectancy_years")[
    ["county", "state", "life_expectancy_years"]
]


The same 65 complete Florida counties appear in every county figure. Two counties are excluded because their primary-care estimates are missing. Unusually low or high values deserve investigation, not automatic removal. County names help us connect those observations to their geographic context.


## Step 3 — Evidence

### Start with a scatterplot

A **scatterplot** places one point for each observation:

- horizontal position represents one quantitative feature;
- vertical position represents the other quantitative feature; and
- the overall pattern shows how the features vary together.


In [ ]:
sns.scatterplot(
    data=analysis,
    x="median_household_income_usd",
    y="life_expectancy_years",
    color="purple"
)
plt.title("Florida County Income and Life Expectancy")
plt.xlabel("Median household income (U.S. dollars)")
plt.ylabel("Life expectancy (years)")
plt.show()


## Discussion: describe the scatterplot

Use four features when describing a relationship between quantitative variables:

1. **Direction:** Does `y` generally increase or decrease as `x` increases?
2. **Form:** Is the main pattern approximately linear, curved, clustered, or absent?
3. **Strength:** How tightly do points follow the main pattern?
4. **Unusual observations:** Which points fall far from the main pattern or far from most x-values?

Describe the relationship between county income and life expectancy using direction, form, strength, and unusual observations. Does the scatterplot support the prediction made earlier?


### Correlation

The **Pearson correlation**, usually written as *r*, summarizes the direction and strength of a **linear** relationship:

- it ranges from -1 through 1;
- positive values indicate a positive direction;
- negative values indicate a negative direction;
- values farther from 0 indicate a stronger linear relationship; and
- correlation has no units and does not establish causation.

Select the two quantitative features in the original relationship before calling `.corr()`.


In [ ]:
analysis[["median_household_income_usd", "life_expectancy_years"]].corr().round(3)

The correlation is approximately 0.744, indicating a moderately strong positive linear association. Inspect the scatterplot as well: a correlation alone cannot establish whether the relationship is approximately straight or whether unusual points influence it.


### Add a regression line

`sns.regplot()` overlays a least-squares line on the observations. Use it to summarize the broad direction of the relationship and see how widely counties vary around it. The shaded band describes uncertainty in the estimated mean line, not the range containing individual counties.


In [ ]:
sns.regplot(
    data=analysis,
    x="median_household_income_usd",
    y="life_expectancy_years",
    color="teal"
)
plt.title("Linear Model of Florida County Income and Life Expectancy")
plt.xlabel("Median household income (U.S. dollars)")
plt.ylabel("Life expectancy (years)")
plt.show()


### R-squared

**R-squared** describes the proportion of observed variation in the response that is accounted for by the fitted model. For a simple linear model with an intercept, R-squared equals the squared correlation.


In [ ]:
r_squared = analysis[["median_household_income_usd", "life_expectancy_years"]].corr().loc[
    "median_household_income_usd", "life_expectancy_years"
] ** 2
round(r_squared, 3)

The model accounts for about 55.4% of the county-level variation in life expectancy. That is substantial, but it also means that about 44.6% of the observed variation is not accounted for by this one-feature line.

R-squared is not the percentage of counties predicted correctly, and it does not measure causation.


### Can another feature add useful information?

Income is not the only county characteristic that may be related to life expectancy. We will examine two possible **covariates**—additional predictor features that may help describe differences among counties with similar incomes:

- the percentage of residents age 65 or older; and
- the number of primary-care physicians per 100,000 residents.

To make the differences easy to visualize, split each feature at its median among the 65 complete Florida counties. The lower group includes the median; the higher group contains values strictly above it. These descriptive cutoffs are specific to this sample, not medical thresholds. Two groups keep the plots readable but discard differences within each group.

Calculate the medians, then use `np.where()` to assign readable labels. `pd.Categorical()` keeps the lower and higher labels in a consistent order for the figures.


In [ ]:
age_median = analysis["age_65_plus_percent"].median()
physician_median = analysis["primary_care_physicians_per_100000"].median()
print("Age 65+ median (%):", round(age_median, 2))
print("Physician median (per 100,000):", round(physician_median, 2))

analysis["age_65_group"] = pd.Categorical(
    np.where(analysis["age_65_plus_percent"] <= age_median,
             "Lower share", "Higher share"),
    categories=["Lower share", "Higher share"], ordered=True
)
analysis["primary_care_group"] = pd.Categorical(
    np.where(analysis["primary_care_physicians_per_100000"] <= physician_median,
             "Lower access", "Higher access"),
    categories=["Lower access", "Higher access"], ordered=True
)


The cutoffs are approximately 21.37% age 65+ and 51.15 physicians per 100,000 residents. Each split has 33 counties at or below the median and 32 above it. The code uses the unrounded medians.


In [ ]:
analysis.head()

Check the number of counties assigned to each category.


In [ ]:
analysis["age_65_group"].value_counts(sort=False)


In [ ]:
analysis["primary_care_group"].value_counts(sort=False)


### Percentage of residents age 65 or older

`sns.lmplot()` displays a scatterplot with fitted regression lines. Setting `hue="age_65_group"` assigns a color to each age category and fits a separate line for each group. These lines help us see whether counties with similar incomes tend to have different life expectancies across age categories. These separate group lines are descriptive visual summaries, not evidence of causation.

`legend=False` prevents `lmplot()` from drawing its automatic legend so we can add one clear legend with a human-readable title. `plt.xticks(rotation=30)` rotates the income labels so large values are easier to distinguish.


In [ ]:
sns.lmplot(
    data=analysis,
    x="median_household_income_usd",
    y="life_expectancy_years",
    hue="age_65_group",
    palette="viridis",
    legend=False
)
plt.title("Florida County Income and Life Expectancy by Age Composition")
plt.xlabel("Median household income (U.S. dollars)")
plt.ylabel("Life expectancy (years)")
plt.xticks(rotation=30)
plt.legend(title="Residents age 65+")
plt.show()


## Discussion: does age composition add information?

At similar income levels, how do the fitted lines differ among counties with lower and higher shares of residents age 65 or older?


### Primary-care access

Repeat the same visual comparison using primary-care physicians per 100,000 residents. Because the plotting structure is identical, the amount of separation among the lines can be compared with the age example.


In [ ]:
sns.lmplot(
    data=analysis,
    x="median_household_income_usd",
    y="life_expectancy_years",
    hue="primary_care_group",
    palette="magma",
    legend=False
)
plt.title("Florida County Income and Life Expectancy by Primary-Care Access")
plt.xlabel("Median household income (U.S. dollars)")
plt.ylabel("Life expectancy (years)")
plt.xticks(rotation=30)
plt.legend(title="Primary-care access")
plt.show()


## Discussion: does primary-care access add information?

How does the separation among these fitted lines compare with the separation in the age-composition figure?


### Compare R-squared values

A categorical feature must be represented numerically before it can enter a linear-model calculation. `pd.get_dummies()` creates 0/1 indicator columns for the categories. `drop_first=True` keeps one category as a reference and avoids storing redundant information.

The `lmplot()` figures fit separate slopes for each group. The comparison below instead uses additive models with one shared income slope and category-specific intercepts, so their fitted lines are not identical.

The function below performs the same least-squares calculation for several predictors and returns R-squared. The matrix calculation is a preview of multiple linear regression; the important result for this lesson is the comparison among models.


In [ ]:
def calculate_r_squared(data, categorical_features):
    predictors = pd.get_dummies(
        data[
            ["median_household_income_usd", *categorical_features]
        ],
        columns=categorical_features,
        drop_first=True,
        dtype=float
    )
    predictors.insert(0, "intercept", 1.0)

    x = predictors.to_numpy()
    y = data["life_expectancy_years"].to_numpy()
    coefficients = np.linalg.lstsq(x, y, rcond=None)[0]
    predicted = x @ coefficients

    residual_sum_squares = ((y - predicted) ** 2).sum()
    total_sum_squares = ((y - y.mean()) ** 2).sum()
    return 1 - residual_sum_squares / total_sum_squares


In [ ]:
model_comparison = pd.Series(
    {
        "Income only": calculate_r_squared(
            analysis, []
        ),
        "Income + age category": calculate_r_squared(
            analysis, ["age_65_group"]
        ),
        "Income + primary-care category": calculate_r_squared(
            analysis, ["primary_care_group"]
        ),
        "Income + both categories": calculate_r_squared(
            analysis,
            ["age_65_group", "primary_care_group"]
        )
    },
    name="r_squared"
)

model_comparison.round(3)


On the same 65 Florida counties, the income-only model has an R-squared of approximately 0.554. Adding age category raises it to 0.609. Adding primary-care category instead raises it to 0.601. Including income and both categories raises it to 0.643.

Age category adds somewhat more information than primary-care category, but both improve the fit to these observed Florida counties. Ordinary training R-squared cannot decrease when predictors are added to a nested least-squares model. A higher value alone does not establish better prediction for new observations; that requires evaluation on held-out data.


## Step 4 — Check

Compare the correlation and the direction of the regression line with the original scatterplot. Revisit the extreme counties displayed during data inspection. Verify that the category labels agree with their original values, including counties exactly at the median. Are the apparent group differences supported where both groups have observations, or only by extrapolating their lines?


In [ ]:
analysis.groupby("age_65_group", observed=True)["age_65_plus_percent"].agg(["min", "max", "count"])


In [ ]:
analysis.groupby("primary_care_group", observed=True)["primary_care_physicians_per_100000"].agg(["min", "max", "count"])


## Step 5 — Conclusion

Among the 65 Florida counties with all four required measures, income and life expectancy have a moderately strong positive, approximately linear association (r ≈ 0.744). The simple line accounts for about 55.4% of observed variation in life expectancy, while counties at similar incomes still differ substantially.

Adding age category raises model R-squared from about 0.554 to 0.609; adding primary-care category instead raises it to 0.601. Including both raises it to 0.643. These comparisons describe fit to the same 65 observed counties; they do not establish causation or improved prediction for new counties.


## Discussion: interpret the county evidence

1. Which features of the scatterplot support describing the association as positive and approximately linear? Where does the line provide an incomplete summary?
2. What do r ≈ 0.744 and R-squared ≈ 0.554 tell us, and what do they not tell us?
3. How do the group plots and model R-squared comparison complement one another? Which added category increases R-squared more in this sample?
4. Why might different category cutoffs or the two excluded counties affect the comparison?
5. Why should these county-level findings not be interpreted as predictions about individual people or as causal effects?


## Step 6 — Limitation

- These are observational county-level associations, not controlled experiments.
- Neither the regression line nor the colored groups establish that changing income, age composition, or physician access causes life expectancy to change.
- County averages do not describe every person; drawing individual conclusions from group-level associations is an **ecological fallacy**.
- Measures in the annual release use different source periods.
- Two Florida counties lack primary-care estimates and are excluded from the figures. Missingness may affect the results.
- Median splits simplify visualization but discard within-group variation; different cutoffs can change the display.
- These findings do not establish performance on new data and should not automatically be generalized beyond Florida or beyond the observed income range.


# When is a straight line not a useful summary?

A line can be drawn even when the pattern is curved. Always inspect the observations for curvature, separated groups, unusual points, changing vertical spread, and weak or absent structure. A large correlation is not enough to choose a straight-line summary.


## A strong relationship that is not linear

EPA fuel-economy data provide a useful contrast. The following file contains non-hybrid gasoline vehicle configurations from model year 2024.


In [ ]:
vehicles = pd.read_csv(
    "data/epa_gasoline_vehicle_configurations_2024.csv"
)

vehicles[["combined_mpg", "co2_grams_per_mile"]].describe().round(1)


Combined MPG and carbon dioxide emitted per mile are strongly related, but the raw-scale pattern is curved: each additional MPG has a larger effect at the low-MPG end than at the high-MPG end.


In [ ]:
sns.regplot(
    data=vehicles,
    x="combined_mpg",
    y="co2_grams_per_mile",
    color="darkorange",
    scatter_kws={"alpha": 0.35}
)
plt.title("A Line Misses the Curved MPG–Emissions Relationship")
plt.xlabel("Combined fuel economy (MPG)")
plt.ylabel("Combined CO₂ emissions (grams per mile)")
plt.show()


In [ ]:
vehicle_correlation = vehicles[
    ["combined_mpg", "co2_grams_per_mile"]
].corr().loc["combined_mpg", "co2_grams_per_mile"]

round(vehicle_correlation, 3)


The correlation is approximately -0.95—stronger in absolute value than the county correlation. Nevertheless, the scatterplot shows that one straight line systematically misses the relationship.


## Discussion: strong correlation, poor linear fit

How can the vehicle correlation be very strong while its linear model is inappropriate?


## When one outlier pulls the line away from the pattern

**Question:** Can one unusual observation make a fitted line a poor summary of most points?

Use dataset III from F. J. Anscombe's (1973) *Graphs in Statistical Analysis*. This is a deliberately constructed teaching example with 11 observations, not a real-world sample; `x` and `y` have no physical units. Ten points lie nearly on a line, while one is far above that pattern. We use this single example to isolate the effect of an outlier.

In [ ]:
outlier_example = pd.read_csv("data/anscombe_selected.csv")
outlier_example = outlier_example.loc[outlier_example["dataset"] == "III"].copy()
outlier_example.head()

In [ ]:
sns.regplot(data=outlier_example, x="x", y="y", ci=None, color="purple")
plt.title("One Outlier Pulls the Fitted Line Away from Most Points")
plt.xlabel("x (constructed values)")
plt.ylabel("y (constructed values)")
plt.show()

### Discussion: inspect the unusual observation

Which point is unusual? How does the line miss the main pattern? Predict how the line would change without that point.

As a sensitivity check, compare the full-data line with a line fit without the point at x = 13. Keep all observations visible. This illustrates influence, not a rule for deleting observations.

In [ ]:
sns.regplot(data=outlier_example, x="x", y="y", ci=None,
            color="purple", line_kws={"label": "All 11 points"})
sns.regplot(data=outlier_example.loc[outlier_example["x"] != 13],
            x="x", y="y", ci=None, scatter=False,
            line_kws={"color": "darkorange", "label": "Sensitivity check: without x = 13"})
plt.title("Check How One Observation Changes the Line")
plt.xlabel("x (constructed values)")
plt.ylabel("y (constructed values)")
plt.legend(handles=plt.gca().lines,
           labels=["All 11 points", "Sensitivity check: without x = 13"])
plt.show()

# Diamonds: revealing a relationship with logarithms

## Question and prediction

**How is diamond price related to carat weight, and does transforming both measurements with logarithms make the relationship look more linear?**

Predict the direction and shape on the original scales. Would doubling carat weight necessarily add the same number of dollars at every starting weight?


## Data

The [ggplot2 diamonds documentation](https://ggplot2.tidyverse.org/reference/diamonds.html) describes 53,940 round-cut diamonds. We use a fixed local copy of the Seaborn example-data CSV, so running the notebook does not require downloading data. Each row is one diamond record.

| Feature used | Meaning | Type |
| --- | --- | --- |
| `carat` | Weight in carats | Quantitative |
| `price` | Price in U.S. dollars | Quantitative |

Other columns describe cut, color, clarity, and dimensions. We do not use those features in this two-feature comparison, and this historical teaching dataset is not a current price list.


In [ ]:
diamonds_df = pd.read_csv("data/diamonds.csv")
diamonds_df.head()


In [ ]:
diamonds_df[["carat", "price"]].isna().sum()


In [ ]:
diamonds_df[["carat", "price"]].describe()


A logarithm requires a positive input. Check both measurements before transforming; missing values, zeros, or negative values need investigation rather than an arbitrary added constant. All 53,940 records have positive, non-missing carat and price values, so the original and transformed plots use exactly the same observations.


In [ ]:
(diamonds_df[["carat", "price"]] <= 0).sum()


## Evidence: original measurements

Plot price against carat weight. Describe direction, form, spread, and unusual observations before transforming.


In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=diamonds_df, x="carat", y="price", color="purple")
plt.title("Diamond Price and Carat Weight: Original Scales")
plt.xlabel("Weight (carats)")
plt.ylabel("Price (U.S. dollars)")
plt.tight_layout()
plt.show()


### Create logarithmic measurements

`np.log10()` takes the base-10 logarithm. It compresses large values: equal distances now represent equal ratios rather than equal differences. For example, prices of $100, $1,000, and $10,000 become 2, 3, and 4. Doubling a positive value always adds about 0.301 to its base-10 logarithm.

Create new columns while preserving the original measurements. We are taking the logarithm of **both** carat and price, not just changing the axis labels. A relationship resembling a power law can appear approximately straight on these log-log coordinates; the resulting line describes the transformed measurements.


In [ ]:
diamonds_df["log10_carat"] = np.log10(diamonds_df["carat"])
diamonds_df["log10_price"] = np.log10(diamonds_df["price"])
diamonds_df.head()


### Transform one feature at a time

Hold one feature on its original scale and transform the other. Use the same records in every figure. Compare the direction and curvature before looking at the plot with both measurements transformed.

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(data=diamonds_df, x="log10_carat", y="price", ci=None,
            line_kws={"color": "darkorange"}, color="purple")
plt.title("Diamond Price and Carat Weight: Log Carat Only")
plt.xlabel("log10(weight in carats)")
plt.ylabel("Price (U.S. dollars)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(data=diamonds_df, x="carat", y="log10_price", ci=None,
            line_kws={"color": "darkorange"}, color="purple")
plt.title("Diamond Price and Carat Weight: Log Price Only")
plt.xlabel("Weight (carats)")
plt.ylabel("log10(price in U.S. dollars)")
plt.tight_layout()
plt.show()

### Transform both features

Now compare the plot with both measurements transformed. Does one straight line follow the main cloud more closely?

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(data=diamonds_df, x="log10_carat", y="log10_price", ci=None,
            line_kws={"color": "darkorange"}, color="purple")
plt.title("Diamond Price and Carat Weight: Log-Log Coordinates")
plt.xlabel("log10(weight in carats)")
plt.ylabel("log10(price in U.S. dollars)")
plt.tight_layout()
plt.show()


The orange line summarizes the transformed relationship; `ci=None` omits an uncertainty band. Compare the cloud's form and vertical spread with the original-scale plot. A straighter appearance does not mean every diamond follows the line.


In [ ]:
pd.Series({
    "Original carat and price": diamonds_df["carat"].corr(diamonds_df["price"]),
    "log10(carat) and log10(price)": diamonds_df["log10_carat"].corr(diamonds_df["log10_price"])
}, name="Pearson correlation").round(3)


## Check

Confirm that the transformation preserves the number of records, produces finite values, and can be reversed for a few original observations. Use the plots to judge form; do not choose a transformation solely because its correlation is larger.


In [ ]:
print("Rows before and after transformation:", len(diamonds_df))
np.isfinite(diamonds_df[["log10_carat", "log10_price"]]).all()


In [ ]:
diamonds_df.head().assign(
    recovered_carat=10 ** diamonds_df["log10_carat"].head(),
    recovered_price=10 ** diamonds_df["log10_price"].head()
).round(3)


## Discussion: compare the four views

1. How do direction, form, and spread change across the original, log-carat-only, log-price-only, and log-log views? Which view has the straightest main pattern?
2. Why does one equal horizontal step in the log plot represent a ratio of weights rather than an equal number of carats?
3. Did the transformation remove any observations or change their original recorded prices?
4. Which features besides carat might help explain the remaining price variation? Why is a straighter plot not proof of causation or an accurate price prediction for every diamond?


## Conclusion and limitation

Heavier diamonds tend to cost more in this dataset. The log-log view makes the relationship between carat and price approximately linear, illustrating how a transformation can reveal structure that is harder to summarize on the original scales.

The transformed axes change how distances are interpreted, not the original records. Remaining scatter, other diamond characteristics, and the dataset's historical scope limit what this two-feature display can explain. A transformation is useful when its meaning and the visual evidence fit the question; it should not be used merely to force a straight-looking result.


## Key takeaways

- Describe a scatterplot's direction, form, strength, and unusual observations before relying on a numerical summary.
- Correlation and simple-line R-squared summarize particular aspects of an association; neither establishes causation.
- A regression plot can summarize a broad trend while leaving substantial variation among observations.
- Colors and group-specific lines reveal additional structure. Comparing nested models on the same rows quantifies changes in training R-squared; improvement on new data requires separate evaluation.
- A single outlier can pull a fitted line away from most observations; investigate it before deciding how to handle it.
- The MPG example shows that a strong correlation can accompany a visibly curved relationship.
- The diamonds example shows how logarithms of positive measurements can make a relationship more nearly linear; equal log distances represent equal ratios.
- Investigate unusual points, preserve meaningful units, and connect conclusions to the observations and analyses actually shown.
